# Notebook 2: Data Collection from OpenAlex

Retrieve metadata for all articles published in finance journals (1918–2020) using the OpenAlex API.

**Outputs:**
- `data/raw/articles.parquet` — one row per article
- `data/raw/authorships.parquet` — one row per author-article pair

**Note:** This notebook takes several hours to run due to API pagination across ~100K+ articles. It saves checkpoints per journal so you can resume if interrupted.

In [ ]:
import pandas as pd
import pyalex
from pyalex import Works
from tqdm.notebook import tqdm
import time
import os
import sys
sys.path.insert(0, ".")
from utils import reconstruct_abstract

# Configure OpenAlex — replace with your email for polite pool (10 req/sec)
pyalex.config.email = "your_email@example.com"

# Paths
CHECKPOINT_DIR = "data/raw/journals"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Libraries loaded.")

In [ ]:
# Load journal list from Notebook 1
journal_list = pd.read_csv("data/processed/journal_list.csv")
print(f"Loaded {len(journal_list)} journals")
journal_list.head()

## Collect Articles from OpenAlex

For each journal, we paginate through all articles (1918–2020) and extract:
- Article metadata (title, year, abstract, journal)
- Authorship details (author name, position, institution, country)

In [ ]:
def collect_journal_articles(source_id: str, journal_name: str) -> tuple[list[dict], list[dict]]:
    """
    Collect all articles from a single journal (1918-2020) via OpenAlex API.
    Returns (articles_list, authorships_list).
    """
    articles = []
    authorships = []
    
    try:
        pager = (
            Works()
            .filter(
                primary_location={"source": {"id": source_id}},
                type="article",
                publication_year="1918-2020",
            )
            .paginate(per_page=200)
        )
        
        for page in pager:
            for work in page:
                work_id = work["id"]
                
                # Reconstruct abstract from inverted index
                abstract = reconstruct_abstract(
                    work.get("abstract_inverted_index")
                )
                
                articles.append({
                    "work_id": work_id,
                    "title": work.get("title"),
                    "publication_year": work.get("publication_year"),
                    "source_id": source_id,
                    "journal_name": journal_name,
                    "abstract": abstract,
                    "doi": work.get("doi"),
                    "cited_by_count": work.get("cited_by_count", 0),
                })
                
                # Extract authorship details
                for auth in work.get("authorships", []):
                    author = auth.get("author", {}) or {}
                    
                    # Get primary institution info
                    institutions = auth.get("institutions", [])
                    inst_name = None
                    country_code = None
                    if institutions:
                        inst = institutions[0]
                        inst_name = inst.get("display_name")
                        country_code = inst.get("country_code")
                    
                    authorships.append({
                        "work_id": work_id,
                        "author_id": author.get("id"),
                        "display_name": author.get("display_name"),
                        "author_position": auth.get("author_position"),
                        "institution_name": inst_name,
                        "country_code": country_code,
                    })
    except Exception as e:
        print(f"\n  Error collecting {journal_name}: {e}")
    
    return articles, authorships


# Collect all journals with checkpointing
all_articles = []
all_authorships = []
skipped = 0

for idx, row in tqdm(journal_list.iterrows(), total=len(journal_list), desc="Journals"):
    source_id = row["source_id"]
    journal_name = row["journal_name"]
    
    # Check for existing checkpoint
    safe_name = journal_name.replace("/", "_").replace(" ", "_")[:60]
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{safe_name}.parquet")
    auth_checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{safe_name}_auth.parquet")
    
    if os.path.exists(checkpoint_path):
        # Load from checkpoint
        arts = pd.read_parquet(checkpoint_path).to_dict("records")
        auths = pd.read_parquet(auth_checkpoint_path).to_dict("records") if os.path.exists(auth_checkpoint_path) else []
        all_articles.extend(arts)
        all_authorships.extend(auths)
        skipped += 1
        continue
    
    # Collect from API
    arts, auths = collect_journal_articles(source_id, journal_name)
    
    # Save checkpoint
    if arts:
        pd.DataFrame(arts).to_parquet(checkpoint_path, index=False)
    if auths:
        pd.DataFrame(auths).to_parquet(auth_checkpoint_path, index=False)
    
    all_articles.extend(arts)
    all_authorships.extend(auths)

if skipped:
    print(f"\n(Loaded {skipped} journals from checkpoints)")
print(f"\nTotal articles collected: {len(all_articles):,}")
print(f"Total authorships collected: {len(all_authorships):,}")

## Save Final Datasets

In [ ]:
# Convert to DataFrames and save
articles_df = pd.DataFrame(all_articles)
authorships_df = pd.DataFrame(all_authorships)

# Remove exact duplicates (can happen with checkpoint reloading)
articles_df = articles_df.drop_duplicates(subset=["work_id"])
authorships_df = authorships_df.drop_duplicates(subset=["work_id", "author_id"])

# Save
articles_df.to_parquet("data/raw/articles.parquet", index=False)
authorships_df.to_parquet("data/raw/authorships.parquet", index=False)

print(f"Articles: {len(articles_df):,}")
print(f"  With abstracts: {articles_df['abstract'].notna().sum():,} ({articles_df['abstract'].notna().mean():.1%})")
print(f"  Year range: {articles_df['publication_year'].min()} - {articles_df['publication_year'].max()}")
print(f"\nAuthorships: {len(authorships_df):,}")
print(f"  Unique authors: {authorships_df['author_id'].nunique():,}")
print(f"\nPaper reference: 125,871 articles from 171 journals")
print(f"\nArticles per journal:")
articles_df["journal_name"].value_counts().head(15)